## Импорты и кконстанты

In [13]:
from pathlib import Path
import pandas as pd
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from PIL import Image


RANDOM_STATE = 42
DATA_DIR = Path("../data")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

## Трансформации

In [14]:
# Статистики ImageNet для нормализации
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# === TRAIN: аугментации и подготовка
train_transforms = transforms.Compose([    
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(0.9, 1.1)), # сохраняет мелкие детали (крошки, пятна), но дает модели видеть тарелку под разными углами и масштабами
    transforms.RandomHorizontalFlip(p=0.5),      # ломаем ориентацию узора
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),        # поворот <- Можно заменить на RandomAffine для более естественных искажений
    transforms.ColorJitter(     # Освещение (блики)
        brightness=0.3,   # ±30% яркости
        contrast=0.3,     # ±30% контраста
        saturation=0.2,   # ±20% насыщенности
        hue=0.05          # лёгкий сдвиг оттенка
    ),        
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))     #Blur (заблюренный фон)  
    ], p=0.15),  # 15% шанс применения. Можно покрутить 0.1 ~ 0.25    
    transforms.ToTensor(),  # обеспечивает правильный формат тензоров [batch_size, каналы(RGB), высота, ширина]
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), # нормализация
])

# === VAL/TEST: только подготовка (без аугментаций)
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

## Dataset для обучения

In [15]:
# torchvision.datasets.ImageFolder автоматически понимает структуру папок:
# data/train/cleaned -> класс 0
# data/train/dirty   -> класс 1
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)

# Смотрим, какие классы он нашел и какие индексы им присвоил
print("Классы:", train_dataset.classes)       # Ожидаем: ['cleaned', 'dirty']
print("Маппинг классов:", train_dataset.class_to_idx) # Ожидаем: {'cleaned': 0, 'dirty': 1}

Классы: ['cleaned', 'dirty']
Маппинг классов: {'cleaned': 0, 'dirty': 1}


## Кастомный Dataset для теста

In [16]:
class TestImageDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        # Читаем submission CSV, чтобы знать порядок ID и общее количество
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # 1. Получаем ID и гарантируем формат с ведущими нулями (например, "0005")
        raw_id = self.df.iloc[idx]['id']
        img_id = str(raw_id).zfill(4)
        
        # 2. Формируем путь с помощью pathlib (оператор /)
        img_path = self.img_dir / f"{img_id}.jpg"
        
        # 3. Загружаем изображение
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            raise FileNotFoundError(f"Изображение не найдено: {img_path}. Проверьте пути и имена файлов.")
        
        # 4. Применяем трансформации
        if self.transform:
            image = self.transform(image)
            
        # 5. Возвращаем тензор и строковый ID (метку не возвращаем, мы её предсказываем)
        return image, img_id
        

### Создание DataLoader

In [17]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

test_dataset = TestImageDataset(SUBMISSION_PATH, TEST_DIR, val_transforms)

test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=8, 
    shuffle=False, # чтобы порядок предсказаний строго совпадал с sample_submission.csv
)

# Проверка до начала инференса
print(f"Всего изображений в тестовом датасете: {len(test_dataset)}")

# Берем первый батч
images_batch, ids_batch = next(iter(test_loader))

print(f"Размер батча изображений: {images_batch.shape}") # Ожидаем: [8, 3, 224, 224] (или меньше для последнего батча)
print(f"Первые 5 ID в батче: {ids_batch[:5]}")
print("Тестовый Dataset и DataLoader работают корректно!")

Всего изображений в тестовом датасете: 744
Размер батча изображений: torch.Size([8, 3, 224, 224])
Первые 5 ID в батче: ('0000', '0001', '0002', '0003', '0004')
Тестовый Dataset и DataLoader работают корректно!
